### Initial Developments
Perform initial developments in notebooks before creating python scripts

In [1]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 1 # fixed seed for reproducible results
CV_FOLDS = 10 # Number of folds for cross validation

### Ingest the dataset and Inspect it

In [2]:
# Load Dataset
data_path = Path("../data/raw/Student_Performance_Teaching_Effectiveness.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "Student_Performance_Teaching_Effectiveness.csv was not found."
    )

raw_students_performance = pd.read_csv(data_path)

print(f"Rows: {raw_students_performance.shape[0]:,}")
print(f"Columns: {raw_students_performance.shape[1]}")

# Validate data for missing or malformed columns
# Inspect schema, nulls, and duplicates
print(raw_students_performance.info())
print("\nNulls:\n", raw_students_performance.isnull().sum())
print("\nDuplicates:", raw_students_performance.duplicated().sum())

# Check Student_Performance distribution
print("\nStudent_Performance distribution:\n", raw_students_performance["Student_Performance"].value_counts(dropna=False))

# Check on your valid inputs selected
# store string in variables to prevent typos
Age = "Age"
Gender = "Gender"
Attendance_Percentage = "Attendance_Percentage"
Assignments_Completed = "Assignments_Completed"
Class_Participation = "Class_Participation"
Study_Hours_per_Week = "Study_Hours_per_Week"
Projects_Submitted = "Projects_Submitted"
required_columns = [
    Age, Gender, Attendance_Percentage, Assignments_Completed, 
    Class_Participation, Study_Hours_per_Week, Projects_Submitted
]

print("\nSummary:\n", raw_students_performance[required_columns].describe(include="all"))

Rows: 1,000
Columns: 15
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Student_ID              1000 non-null   str    
 1   Age                     1000 non-null   int64  
 2   Gender                  1000 non-null   str    
 3   Attendance_Percentage   1000 non-null   float64
 4   Assignments_Completed   1000 non-null   int64  
 5   Class_Participation     1000 non-null   float64
 6   Midterm_Grade           1000 non-null   float64
 7   Final_Grade             1000 non-null   float64
 8   Study_Hours_per_Week    1000 non-null   int64  
 9   Projects_Submitted      1000 non-null   int64  
 10  Teaching_Clarity        1000 non-null   float64
 11  Teaching_Helpfulness    1000 non-null   float64
 12  Feedback_Timeliness     1000 non-null   float64
 13  Student_Performance     1000 non-null   str    
 14  Teaching_Effectiveness  1000

### Feature Engineering and Validation

In [3]:
Student_Performance = "Student_Performance" # Target response
ALLOWED_GENDERS = {"Male", "Female", "Other"}
ALLOWED_STUDENT_PERFORMANCE = {"Excellent", "Good", "Average", "Poor"}

def validate_data(df: pd.DataFrame, is_training: bool = True) -> pd.DataFrame:
    expected_cols = required_columns + ([Student_Performance] if is_training else [])
    
    # Missing columns
    missing = set(expected_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Nulls & duplicates
    if df[expected_cols].isnull().any().any():
        raise ValueError("Null values detected in required columns.")
    if df.duplicated().any():
        df = df.drop_duplicates().copy()

    # Expected categories
    if not set(df[Gender].unique()).issubset(ALLOWED_GENDERS):
        raise ValueError(f"Unexpected Gender values: {set(df[Gender].unique()) - ALLOWED_GENDERS}")
    if is_training and not set(df[Student_Performance].unique()).issubset(ALLOWED_STUDENT_PERFORMANCE):
        raise ValueError("Unexpected target categories detected.")

    # Valid numeric ranges
    range_checks = {
        Age: (18, 100),
        Attendance_Percentage: (0, 100),
        Assignments_Completed: (0, 10),
        Class_Participation: (0, 10),
        Study_Hours_per_Week: (0, 168),
        Projects_Submitted: (0, 5),
    }
    for col, (low, high) in range_checks.items():
        if not df[col].between(low, high).all():
            raise ValueError(f"Column '{col}' has values outside [{low}, {high}].")

    print("Validation passed.")
    return df

def add_student_work_engagement_features(df: pd.DataFrame) -> pd.DataFrame:
    # Keep only required inputs
    df_out = df[required_columns].copy()
    
    # Coursework completion based on an average of assignments and projects submitted
    df_out["Coursework_Completion_Rate"] = ((df_out[Assignments_Completed] / 10.0) + (df_out[Projects_Submitted] / 5.0)) / 2.0
    
    # Engagement Score
    df_out["Engagement_Score"] = ((df_out[Attendance_Percentage] / 100.0) * df_out[Class_Participation])
    return df_out

# Validate and add new features
validated_df = validate_data(raw_students_performance, is_training=True)
X = add_student_work_engagement_features(validated_df)
y = validated_df[Student_Performance]

Validation passed.


### Training Pipeline

In [4]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Compatibility for older and newer scikit-learn versions.
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

# Two engineered features
Coursework_Completion_Rate="Coursework_Completion_Rate"
Engagement_Score="Engagement_Score"
numeric_features = [
    Age, Attendance_Percentage, Assignments_Completed, 
    Class_Participation, Study_Hours_per_Week, Projects_Submitted,
    Coursework_Completion_Rate, Engagement_Score
]

categorical_features = [Gender]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot)
])

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)


def build_model_pipeline(estimator):
    """Attach a candidate estimator to the fixed feature pipeline."""
    return Pipeline(steps=[
        ("feature_engineering", FunctionTransformer(add_student_work_engagement_features, validate=False)),
        ("preprocess", clone(preprocess)),
        ("model", estimator)
    ])

### Train-test Split
80% - training \
20% - testing

In [13]:

from sklearn.model_selection import train_test_split

X_raw = validated_df[required_columns]
y = validated_df[Student_Performance]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    stratify=y, # Divides Student Performance proportionally for the train-test split
    random_state=42,
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print("\nStudent Performance size")
print("Train Student Performance:\n", y_train.value_counts(dropna=False))
print("\nTest Student Performance:\n", y_test.value_counts(dropna=False))
print("\nStudent Performance proportions")
print("Train Student Performance:\n", y_train.value_counts(normalize=True))
print("\nTest Student Performance:\n", y_test.value_counts(normalize=True))

Train shape: (800, 7)
Test shape: (200, 7)

Student Performance size
Train Student Performance:
 Student_Performance
Good         330
Average      233
Excellent    162
Poor          75
Name: count, dtype: int64

Test Student Performance:
 Student_Performance
Good         83
Average      58
Excellent    41
Poor         18
Name: count, dtype: int64

Student Performance proportions
Train Student Performance:
 Student_Performance
Good         0.41250
Average      0.29125
Excellent    0.20250
Poor         0.09375
Name: proportion, dtype: float64

Test Student Performance:
 Student_Performance
Good         0.415
Average      0.290
Excellent    0.205
Poor         0.090
Name: proportion, dtype: float64


### 10-Fold Cross Validation
Perform 10-Fold CV on different models

In [ ]:
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_validate

# Set up local SQLite MLflow tracking
db_path = (Path("..") / "mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("student_performance_classification")

# Experiment configuration
CONFIG = {
    "dataset_version": "Student_Performance_Teaching_Effectiveness_v1.csv",
    "preprocessing_version": "v1_median_impute_scaler_onehot",
    "feature_set_version": "v1_6_raw_2_engineered",
    "test_size": 0.2,
    "cv_folds": CV_FOLDS,
    "random_state": RANDOM_STATE,
    "target": Student_Performance,
    "features": numeric_features + categorical_features,
}

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
tree_count = 100

candidates = {
    "Dummy (Most Frequent)": build_model_pipeline(
        DummyClassifier(strategy="most_frequent")
    ),
    "Decision Tree": build_model_pipeline(
        DecisionTreeClassifier(
            class_weight="balanced",  # Account for low proportions of "Poor" Student performance
            random_state=RANDOM_STATE,
            max_depth=8,
            min_samples_leaf=20,
        )
    ),
    "Random Forest": build_model_pipeline(
        RandomForestClassifier(
            n_estimators=tree_count,
            class_weight="balanced",  # Account for low proportions of "Poor" Student performance
            random_state=RANDOM_STATE,
            max_depth=12,
            min_samples_leaf=3,
            n_jobs=-1,
        )
    ),
}

cv_rows = []
for name, pipe in candidates.items():
    with mlflow.start_run(run_name=name):
        # 1. Model, preprocessing & dataset versions
        mlflow.set_tags({
            "model_version": f"{name}_v1",
            "preprocessing_version": CONFIG["preprocessing_version"],
            "dataset_version": CONFIG["dataset_version"],
        })

        # Configuration artifact
        mlflow.log_dict(CONFIG, "config.json")

        # Parameters
        estimator = pipe.named_steps["model"]
        params = {
            "model_name": name,
            "cv_folds": CV_FOLDS,
            "random_state": RANDOM_STATE,
        }
        for param in ["strategy", "class_weight", "n_estimators", "max_depth", "min_samples_leaf"]:
            if hasattr(estimator, param):
                params[param] = getattr(estimator, param)
        mlflow.log_params(params)

        # Evaluation metrics (CV)
        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring=["f1_macro", "accuracy"],
            n_jobs=-1,
        )

        f1_mean = scores["test_f1_macro"].mean()
        f1_std = scores["test_f1_macro"].std()
        acc_mean = scores["test_accuracy"].mean()
        acc_std = scores["test_accuracy"].std()

        mlflow.log_metrics({
            "cv_f1_macro_mean": f1_mean,
            "cv_f1_macro_std": f1_std,
            "cv_accuracy_mean": acc_mean,
            "cv_accuracy_std": acc_std,
        })

        # Fit on full train split and log model artifact
        pipe.fit(X_train, y_train)
        mlflow.sklearn.log_model(
            pipe,
            name="model",
            skops_trusted_types=[
                "__main__.add_student_work_engagement_features",
                "numpy.dtype",
                "sklearn.tree._tree.Tree",
            ],
        )

        cv_rows.append({
            "Model": name,
            "CV Macro F1 (mean)": round(f1_mean, 4),
            "CV Macro F1 (std)": round(f1_std, 4),
            "CV Accuracy (mean)": round(acc_mean, 4),
            "CV Accuracy (std)": round(acc_std, 4),
        })

cv_summary = pd.DataFrame(cv_rows).sort_values("CV Macro F1 (mean)", ascending=False)
print(cv_summary.to_string(index=False))

2026/11/27 04:27:32 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp20pogu5n/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.1', 'skops==0.16.0']. Set logging level to DEBUG to see the full traceback. 
2026/11/27 04:27:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/27 04:27:34 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpq4u6k9da/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.1', 'skops==0.16.0']. Set logging level to DEBUG to see the full traceback. 
2026/11/27 04:27:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/27 04:27:37 WARNING mlflow.utils.e

                Model  CV Macro F1 (mean)  CV Macro F1 (std)  CV Accuracy (mean)  CV Accuracy (std)
        Random Forest              0.2544             0.0220              0.2950             0.0317
        Decision Tree              0.2046             0.0349              0.2250             0.0411
Dummy (Most Frequent)              0.1456             0.0122              0.4125             0.0474
